# EEG_07 — Pre-calcolo Grafi per Trial

Calcola e salva su disco gli  per tutti i trial e tutti i metodi:
- **PCC** — Pearson Correlation Coefficient
- **PLV** — Phase Locking Value (theta + alpha)
- **wPLI** — Weighted Phase Lag Index

Output: 
Formato: 

> Esegui questo notebook UNA VOLTA. EEG_08/09/10 caricheranno la cache da disco.
> I metodi "learned" e "dynamic" non richiedono pre-calcolo (grafo appreso/calcolato su GPU).

In [ ]:
# ============================================================
# CONFIG — deve essere identica a EEG_08
# ============================================================
from pathlib import Path
import numpy as np
import torch
import h5py
import pandas as pd
from tqdm import tqdm
from scipy.signal import butter, filtfilt
from scipy.signal import hilbert as sp_hilbert

project_root = next(
    (p for p in [Path().resolve()] + list(Path().resolve().parents)
     if (p / ".git").exists()),
    Path().resolve()
)

META_CSV  = project_root / "data" / "interim" / "eeg_metadata.csv"
ELOC_PATH = project_root / "src" / "io" / "ebneuro.locs"
GRAPHS_DIR = project_root / "data" / "interim" / "graphs"
GRAPHS_DIR.mkdir(parents=True, exist_ok=True)

K_GRAPH   = 6       # vicini k-NN (deve coincidere con EEG_08)
SFREQ     = 256     # Hz
N_CHANS   = 59
PLV_BANDS = [(4, 8), (8, 13)]   # theta + alpha
# PLV_BANDS = [(4, 8), (8, 13), (13, 30), (30, 80)]  # + beta + gamma

METHODS   = ["pcc", "plv", "wpli"]   # learned e dynamic non servono pre-calcolo

print(f"project_root: {project_root}")
print(f"Output dir:   {GRAPHS_DIR}")
print(f"Metodi:       {METHODS}")
print(f"PLV_BANDS:    {PLV_BANDS}")

In [ ]:
# ============================================================
# METADATA + CANALI
# ============================================================
meta = pd.read_csv(META_CSV)

# Filtro righe corrotte
meta = meta[~(
    (meta["path_h5"].str.contains("08_05.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_01.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_03.h5") & (meta["epoch_idx"] >= 110)) |
    (meta["path_h5"].str.contains("46_04.h5") & (meta["epoch_idx"] >= 34))
)]
meta["subject_id"] = meta["subject_id"].astype(str).str.zfill(2)

def read_eloc_names(path):
    names = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                names.append(parts[3])
    return names[:61]

ch_names_61 = read_eloc_names(ELOC_PATH)
EXCLUDE     = {"A1", "A2"}
keep_idx    = [i for i, n in enumerate(ch_names_61) if n not in EXCLUDE]
assert len(keep_idx) == N_CHANS

print(f"Record totali: {len(meta)}")
print(f"Canali: {len(keep_idx)}")

In [ ]:
# ============================================================
# FUNZIONI DI COSTRUZIONE GRAFO
# ============================================================

def knn_from_matrix(matrix, k):
    edges = set()
    for i in range(matrix.shape[0]):
        for j in np.argsort(matrix[i])[::-1][:k]:
            edges.add((i, int(j)))
            edges.add((int(j), i))
    src, dst = zip(*sorted(edges))
    return torch.tensor([list(src), list(dst)], dtype=torch.long)


def pcc_to_edge_index(x_np, k=6):
    pcc = np.abs(np.corrcoef(x_np))
    np.fill_diagonal(pcc, 0.0)
    return knn_from_matrix(pcc, k)


def plv_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    plv_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt   = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)
        exp_phi  = np.exp(1j * np.angle(analytic))
        plv      = np.abs(exp_phi @ exp_phi.conj().T) / x_np.shape[1]
        plv_sum += plv
    plv_matrix = (plv_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(plv_matrix, 0.0)
    return knn_from_matrix(plv_matrix, k)


def wpli_to_edge_index(x_np, k=6, sfreq=256, bands=None):
    if bands is None:
        bands = PLV_BANDS
    nyq = sfreq / 2.0
    wpli_sum = np.zeros((x_np.shape[0], x_np.shape[0]), dtype=np.float64)
    for flo, fhi in bands:
        b, a = butter(4, [flo / nyq, fhi / nyq], btype="band")
        x_filt   = filtfilt(b, a, x_np, axis=1).astype(np.float32)
        analytic = sp_hilbert(x_filt, axis=1)         # (C, T) complex
        # Cross-spettro: C_ij(t) = analytic_i(t) * conj(analytic_j(t))
        # imag shape: (C, C, T)
        imag_cs  = np.imag(
            analytic[:, :, np.newaxis] * analytic[np.newaxis, :, :].conj()
        )
        wpli_band = np.abs(imag_cs.mean(axis=-1)) / (np.abs(imag_cs).mean(axis=-1) + 1e-8)
        wpli_sum += wpli_band
    wpli_matrix = (wpli_sum / len(bands)).astype(np.float32)
    np.fill_diagonal(wpli_matrix, 0.0)
    return knn_from_matrix(wpli_matrix, k)


GRAPH_FN = {
    "pcc":  pcc_to_edge_index,
    "plv":  plv_to_edge_index,
    "wpli": wpli_to_edge_index,
}

print("Funzioni grafo OK")

In [ ]:
# ============================================================
# PRE-CALCOLO — itera tutti i trial per ogni metodo
# ============================================================
# Output per ogni metodo:
#   GRAPHS_DIR / f"{method}_k{K_GRAPH}.pt"
#   -> dict: {(subject_id_str, epoch_idx_int): edge_index_tensor}

records = meta[["path_h5", "epoch_idx", "subject_id"]].to_dict("records")

for method in METHODS:
    out_path = GRAPHS_DIR / f"{method}_k{K_GRAPH}.pt"
    if out_path.exists():
        print(f"[{method}] già calcolato → {out_path.name}  (skip)")
        continue

    print(f"
[{method}] Calcolo {len(records)} grafi...")
    fn       = GRAPH_FN[method]
    cache    = {}
    h5_cache = {}

    for r in tqdm(records, desc=method):
        path    = r["path_h5"]
        e_idx   = int(r["epoch_idx"])
        subj    = str(r["subject_id"])
        key     = (subj, e_idx)

        if path not in h5_cache:
            h5_cache[path] = h5py.File(path, "r")
        x_np = h5_cache[path]["data"][e_idx][keep_idx, :].astype(np.float32)

        cache[key] = fn(x_np, k=K_GRAPH)

    # Chiudi tutti gli h5 aperti
    for f in h5_cache.values():
        f.close()

    torch.save(cache, out_path)
    size_mb = out_path.stat().st_size / 1e6
    print(f"[{method}] Salvato: {out_path.name}  ({len(cache)} grafi, {size_mb:.1f} MB)")

print("
Pre-calcolo completato.")

In [ ]:
# ============================================================
# VERIFICA — controlla i file salvati
# ============================================================
print("File salvati in", GRAPHS_DIR)
for method in METHODS:
    out_path = GRAPHS_DIR / f"{method}_k{K_GRAPH}.pt"
    if out_path.exists():
        cache    = torch.load(out_path, weights_only=False)
        keys     = list(cache.keys())
        sample_ei = cache[keys[0]]
        size_mb  = out_path.stat().st_size / 1e6
        print(f"  {method}_k{K_GRAPH}.pt  |  {len(cache)} grafi  |  {size_mb:.1f} MB")
        print(f"    esempio edge_index shape: {sample_ei.shape}  (2, E)")
    else:
        print(f"  {method}_k{K_GRAPH}.pt  — NON TROVATO")